<h4> Zadanie1 (1 pkt) Rozważ dane *ecoli.txt*. Celem tego zadania jest klasyfikacja lokalizacji subkomórkowej białek na podstawie określonych cech wyprowadzonych z ich sekwencji aminokwasowych. Cechy te reprezentują właściwości biochemiczne i strukturalne, które mogą pomóc w określeniu, w której części komórki dane białko najprawdopodobniej się znajduje.

- Podziel dane na zbiory treningowy i testowy (proporcja 7:3).
- Zbuduj trzy klasyfikatory, oparte na metodach: Regresji logistycznej, SVM oraz Drzewo decyzyjne (parametry domyślne lub własne); dane nie są zbalansowane (pamiętaj o parametrze: class_weight='balanced', być może warto zastosować, a być może nie)
- Porównaj dokładność (accuracy) oraz inne wskaźniki poszczególnych klasyfikatorów na zbiorze testowym.
- Które cechy okazały się najbardziej przydatne do klasyfikacji? (możesz wykorzystać las losowy lub popatrz na wagi modelu w przypadku regresji logistycznej czy liniowego SVM)

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
df = pd.read_csv("ecoli.txt", index_col=0)


In [24]:
from collections import Counter
print(Counter(df["class"]))

Counter({'cp': 143, 'im': 77, 'pp': 52, 'imU': 35, 'om': 20, 'omL': 5, 'imS': 2, 'imL': 2})


In [25]:
dff = df[~df['class'].isin(["omL","imS",'imL'])] #usunę najmniej liczne klasy
print(dff)

X = dff.drop("class", axis = 1)
y = dff["class"]

                mcg   gvh   lip  chg   aac  alm1  alm2 class
sequence_name                                               
AAT_ECOLI      0.49  0.29  0.48  0.5  0.56  0.24  0.35    cp
ACEA_ECOLI     0.07  0.40  0.48  0.5  0.54  0.35  0.44    cp
ACEK_ECOLI     0.56  0.40  0.48  0.5  0.49  0.37  0.46    cp
ACKA_ECOLI     0.59  0.49  0.48  0.5  0.52  0.45  0.36    cp
ADI_ECOLI      0.23  0.32  0.48  0.5  0.55  0.25  0.35    cp
...             ...   ...   ...  ...   ...   ...   ...   ...
TREA_ECOLI     0.74  0.56  0.48  0.5  0.47  0.68  0.30    pp
UGPB_ECOLI     0.71  0.57  0.48  0.5  0.48  0.35  0.32    pp
USHA_ECOLI     0.61  0.60  0.48  0.5  0.44  0.39  0.38    pp
XYLF_ECOLI     0.59  0.61  0.48  0.5  0.42  0.42  0.37    pp
YTFQ_ECOLI     0.74  0.74  0.48  0.5  0.31  0.53  0.52    pp

[327 rows x 8 columns]


In [26]:
# 1. Podział danych na zbiory treningowy i testowy (proporcja 7:3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y) #stratify dla zachowania proporcji klas 

In [27]:
# Budowa trzech klasyfikatorów (z uwzględnieniem niezbalansowanych klas)
log_reg = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
svm_clf = SVC(class_weight='balanced', random_state=42) 
tree_clf = DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [28]:
# Trenowanie modeli
log_reg.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)
tree_clf.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [29]:
# Porównanie wskaźników
models = {
    "Regresja Logistyczna": log_reg,
    "SVM": svm_clf,
    "Drzewo Decyzyjne": tree_clf
}

In [30]:
# Wyniki
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f" Model: {name}")
    print(f"Dokładność (Accuracy): {accuracy_score(y_test, y_pred):.4f}")

    print(classification_report(y_test, y_pred))

 Model: Regresja Logistyczna
Dokładność (Accuracy): 0.8990
              precision    recall  f1-score   support

          cp       0.93      0.98      0.95        43
          im       0.93      0.61      0.74        23
         imU       0.65      1.00      0.79        11
          om       1.00      1.00      1.00         6
          pp       1.00      1.00      1.00        16

    accuracy                           0.90        99
   macro avg       0.90      0.92      0.90        99
weighted avg       0.92      0.90      0.90        99

 Model: SVM
Dokładność (Accuracy): 0.9091
              precision    recall  f1-score   support

          cp       0.96      1.00      0.98        43
          im       0.94      0.65      0.77        23
         imU       0.65      1.00      0.79        11
          om       1.00      1.00      1.00         6
          pp       1.00      0.94      0.97        16

    accuracy                           0.91        99
   macro avg       0.91      0

In [31]:
# Las losowy do oceny przydatności cech
rf_clf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_clf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
# Zbieranie wag
feat_labels = X.columns
forest = RandomForestClassifier(n_estimators=1000, class_weight='balanced', random_state=2)
forest.fit(X_train, y_train)
importances = forest.feature_importances_
indices = np.argsort(importances)[::-1]

In [33]:
for f in range(X_train.shape[1]):
    print(f"{f+1:2d}) {feat_labels[indices[f]]:<10} {round(importances[indices[f]], 4)}")

 1) alm1       0.2269
 2) aac        0.2178
 3) mcg        0.2101
 4) alm2       0.172
 5) gvh        0.167
 6) lip        0.0062
 7) chg        0.0


<h4> Zadanie2/Mini projekt (3 pkt) Przewidywanie czy dana reszta aminokwasowa (K, P, R lub T) jest karbonylowana.
Rozważ dane:
    
- https://doi.org/10.1371/journal.pone.0111478.s003
- https://doi.org/10.1371/journal.pone.0111478.s004

A następnie:
- Wybierz co najmniej pięć właściwości fizykochemicznych aminokwasów.
- Przedstaw każdą sekwencję jako wektor oparty na wybranych właściwościach jej aminokwasów - dla każdej litery w sekwencji + różne właściowości fizykochemiczne.
- (Ewentualnie) Zmniejsz liczbę cech, wykorzystując PCA lub F-score.
- Przeprowadź standaryzację danych przed trenowaniem.
- Zbuduj model w oparciu o drzewa decyzyjne/las losowy; pamiętaj o dostrojeniu jego hiperparametrów (np. GridSearch).
- Oceń model na danych testowych, używając różnych miar wydajności — oceniając osobno K, P, R i T (np. classification report).
- W przypadku uzyskania niesatysfakcjonujących miar oceny modelu (np. dla wybranej kategorii wskaźniki są bardzo niskie) sróbuj wybrać inne cechy (reprezentacje), hiperparametry itd... (pokaż proces ulepszania modelu)
- Opracuj to zadanie w fomie 1-stronnicowego sprawozdania/podsumowania: wybrane reprezentacje sekwencje, jak i finalny model, jakie wyniki na finalnym modelu

In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


In [61]:
# Wczytanie danych s004
df_raw = pd.read_excel("pone.0111478.s004.xlsx", header=None)

# Mapowanie kolumn
cols_map = {0:('K',1), 1:('K',0), 2:('R',1), 3:('R',0), 4:('T',1), 5:('T',0), 6:('P',1), 7:('P',0),
            9:('K',1), 10:('K',0), 11:('R',1), 12:('R',0), 13:('T',1), 14:('T',0), 15:('P',1), 16:('P',0)}

data_list = []
for col_idx, (aa_type, label) in cols_map.items():
    # Wybieramy kolumnę od 6 wiersza w dół
    col_data = df_raw.iloc[6:, col_idx]
    # Odpowiednik iloc[1::2], bierze sekwencje a pomija nagłówki
    sequences = col_data.iloc[1::2].dropna()
    
    for s in sequences:
        if len(str(s)) == 27:
            data_list.append({'seq': str(s), 'label': label, 'type': aa_type})

df = pd.DataFrame(data_list)

In [63]:
# 5 właściwości dla każdego aminokwasu: [Masa, Hydrofobowość, pI, Polarność, Objętość]
aa_properties = {
    'A': [89.1, 1.8, 6.00, 8.1, 67.5], 'C': [121.2, 2.5, 5.02, 5.5, 86.0],
    'D': [133.1,-3.5, 2.77, 13.0, 91.0], 'E': [147.1,-3.5, 3.22, 12.3, 109.0],
    'F': [165.2, 2.8, 5.48, 5.2, 135.0], 'G': [75.1,-0.4, 5.97, 9.0, 48.0],
    'H': [155.2,-3.2, 7.59, 10.4, 118.0], 'I': [131.2, 4.5, 6.02, 5.2, 124.0],
    'K': [146.2,-3.9, 9.74, 11.3, 135.0], 'L': [131.2, 3.8, 5.98, 4.9, 124.0],
    'M': [149.2, 1.9, 5.74, 5.7, 124.0], 'N': [132.1,-3.5, 5.41, 11.6, 96.0],
    'P': [115.1,-1.6, 6.30, 8.0, 90.0], 'Q': [146.1,-3.5, 5.65, 10.5, 114.0],
    'R': [174.2,-4.5,10.76, 10.5, 148.0], 'S': [105.1,-0.8, 5.68, 9.2, 73.0],
    'T': [119.1,-0.7, 5.60, 8.6, 93.0], 'V': [117.1, 4.2, 5.96, 5.9, 105.0],
    'W': [204.2,-0.9, 5.89, 5.4, 163.0], 'Y': [181.2,-1.3, 5.66, 6.2, 141.0]
}

X = np.array([[val for char in s for val in aa_properties#.get(char.upper(), [0]*5)] for s in df['seq']])
y = df['label'].values

In [64]:
# Analiza
X_train, X_test, y_train, y_test, type_train, type_test = train_test_split(
    X, y, df['type'].values, test_size=0.3, random_state=1, stratify=y
)

sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.transform(X_test)

# F-score 
sel = SelectKBest(f_classif, k=30)
X_train_sel = sel.fit_transform(X_train_std, y_train)
X_test_sel = sel.transform(X_test_std)

# Model
params = {'n_estimators': [100, 200], 'max_depth': [10, None]}
grid = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=1), params, cv=3)
grid.fit(X_train_sel, y_train)

,estimator,RandomForestC...andom_state=1)
,param_grid,"{'max_depth': [10, None], 'n_estimators': [100, 200]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [66]:
# WYNIKI
print(classification_report(y_test, grid.predict(X_test_sel)))

# Osobna ocena dla K, P, R, T 
for t in ['K', 'P', 'R', 'T']:
    mask = (type_test == t)
    print(f"\nWyniki dla {t}:")
    print(classification_report(y_test[mask], grid.predict(X_test_sel)[mask], zero_division=0))

              precision    recall  f1-score   support

           0       0.86      1.00      0.93      1296
           1       0.00      0.00      0.00       205

    accuracy                           0.86      1501
   macro avg       0.43      0.50      0.46      1501
weighted avg       0.75      0.86      0.80      1501


Wyniki dla K:
              precision    recall  f1-score   support

           0       0.84      1.00      0.92       573
           1       0.00      0.00      0.00       106

    accuracy                           0.84       679
   macro avg       0.42      0.50      0.46       679
weighted avg       0.71      0.84      0.77       679


Wyniki dla P:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94       232
           1       0.00      0.00      0.00        30

    accuracy                           0.89       262
   macro avg       0.44      0.50      0.47       262
weighted avg       0.78      0.89      0.83 

/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"